# Volumen de fotos: escaneo y base de datos maestra

Este notebook corre directo contra `Y:\Galería` (unidad de red, requiere estar dentro de ITESO). No necesita exportar nada a Excel primero, a diferencia de SharePoint: como es una unidad de archivos real, Python puede leer nombres, tamaños y fechas directamente.

**Flujo de esta primera etapa:**
1. Escanear `FOTOS` y `ORQUIDEAS_GUIA` completos, sin interpretar nada, a la tabla `staging_escaneo`.
2. Sobre esa tabla, intentar adivinar autor/especie a partir de la ruta (path parsing), marcando el nivel de confianza.
3. Exportar a Excel un reporte de qué quedó en alta/media/baja confianza, para que lo revises antes de que decidamos el triage final hacia las tablas normalizadas.

**Todavía NO se llenan `autores`, `especies`, `eventos` ni `fotos`.** Eso lo hacemos en una siguiente pasada, una vez que veas estos resultados y confirmemos ajustes al parser, igual que hicimos con la taxonomía de SharePoint.

In [ ]:
import os
import re
import sqlite3
from pathlib import Path
from datetime import datetime

import json

def cargar_config(archivo="../config_local.json"):
    ruta = Path(archivo)
    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encontro {archivo}. Copia config_local.example.json a {archivo} "
            "(junto a este notebook) y ajusta tus rutas locales. No se sube a git."
        )
    return json.loads(ruta.read_text(encoding="utf-8"))

config = cargar_config()
RAIZ_GALERIA = Path(config["raiz_galeria"])
CARPETAS_A_ESCANEAR = ["FOTOS", "ORQUIDEAS_GUIA"]
RUTA_BD = Path("../bd") / "campuslab_volumen.db"
RUTA_BD.parent.mkdir(exist_ok=True)

print(f"Base de datos: {RUTA_BD.resolve()}")
for c in CARPETAS_A_ESCANEAR:
    ruta = RAIZ_GALERIA / c
    print(f"{'✓' if ruta.exists() else '✗ NO ENCONTRADA'}  {ruta}")

In [ ]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS staging_escaneo (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ruta_completa TEXT NOT NULL UNIQUE,
    nombre_archivo TEXT NOT NULL,
    extension TEXT,
    carpeta_contenedora TEXT,
    rama_raiz TEXT,
    tamano_bytes INTEGER,
    fecha_creacion TEXT,
    fecha_modificacion TEXT,
    autor_sugerido TEXT,
    especie_cientifico_sugerido TEXT,
    especie_comun_sugerido TEXT,
    breadcrumb_especie TEXT,
    confianza_autor TEXT,
    confianza_especie TEXT,
    procesado INTEGER DEFAULT 0,
    fecha_escaneo TEXT DEFAULT CURRENT_TIMESTAMP
);
CREATE TABLE IF NOT EXISTS autores (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL UNIQUE,
    iniciales TEXT,
    notas TEXT
);
CREATE TABLE IF NOT EXISTS especies (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre_cientifico TEXT UNIQUE,
    nombre_comun TEXT,
    grupo TEXT,
    sugerido_por TEXT,
    fuente_validacion TEXT,
    id_externo_fuente TEXT,
    notas TEXT
);
CREATE TABLE IF NOT EXISTS eventos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL UNIQUE,
    anio INTEGER,
    notas TEXT
);
CREATE TABLE IF NOT EXISTS espacios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL UNIQUE,
    notas TEXT
);
CREATE TABLE IF NOT EXISTS fotos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ruta_actual TEXT NOT NULL UNIQUE,
    ruta_original TEXT,
    autor_id INTEGER REFERENCES autores(id),
    especie_id INTEGER REFERENCES especies(id),
    evento_id INTEGER REFERENCES eventos(id),
    espacio_id INTEGER REFERENCES espacios(id),
    tomada_en_iteso INTEGER,
    estado TEXT DEFAULT 'pendiente',
    tamano_bytes INTEGER,
    fecha_alta TEXT DEFAULT CURRENT_TIMESTAMP,
    origen TEXT
);
"""

con = sqlite3.connect(RUTA_BD)
con.executescript(SCHEMA)
con.commit()

# Compatibilidad con bases creadas antes de agregar estas columnas (como esta, si ya existía)
cols_especies = [r[1] for r in con.execute("PRAGMA table_info(especies)")]
for col in ("sugerido_por", "fuente_validacion", "id_externo_fuente"):
    if col not in cols_especies:
        con.execute(f"ALTER TABLE especies ADD COLUMN {col} TEXT")
con.commit()

print("Esquema listo:", [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")])

In [ ]:
CONECTORES = {"de", "del", "la", "las", "los", "y"}

def parece_nombre_valido(s):
    for palabra in s.split():
        if palabra.lower() in CONECTORES:
            continue
        if not re.match(r"^[A-ZÁÉÍÓÚÑ][a-záéíóúñ]*$", palabra):
            return False
    return True

def parsear_autor(nombre_carpeta):
    s = nombre_carpeta.strip()
    m = re.match(r"^([A-Z]{2,4})[_\s]+(?:Fotos\s+)?(.+)$", s)
    if m and len(m.group(2).split()) >= 2 and parece_nombre_valido(m.group(2)):
        return m.group(2).strip(), "alta"
    m = re.match(r"^Fotos\s+(.+)$", s, re.IGNORECASE)
    if m and len(m.group(1).split()) >= 2 and parece_nombre_valido(m.group(1)):
        return m.group(1).strip(), "alta"
    if len(s.split()) >= 2 and parece_nombre_valido(s):
        return s, "alta"
    return s, "baja"

def parsear_especie(nombre):
    s = nombre.strip()
    m = re.match(r"^([A-ZÁÉÍÓÚ][a-záéíóú]+ [a-záéíóú]+)\s*-\s*(.+)$", s)
    if m:
        return m.group(1), m.group(2), "alta"
    m = re.match(r"^[A-ZÁÉÍÓÚ][a-záéíóú]+ [a-záéíóú]+$", s)
    if m:
        return s, None, "alta"
    m = re.match(r"^(.+?)(\d)$", s)
    if m:
        return None, m.group(1), "media"
    return None, s, "media"

def carpeta_autor_desde_partes(partes_ruta):
    """Sube hasta el primer nivel debajo de AUTORES, sin importar qué tan
    profundo viva el archivo (a diferencia de usar solo la carpeta inmediata,
    que falla cuando hay subcarpetas intermedias tipo 'Revisadas' o un taller)."""
    try:
        i = [p.upper() for p in partes_ruta].index("AUTORES")
    except ValueError:
        return None
    if i + 1 < len(partes_ruta) - 1:  # -1: el último elemento es el nombre del archivo
        return partes_ruta[i + 1]
    return None

def breadcrumb_desde_especie(partes_ruta):
    """Todas las carpetas entre ESPECIE y el archivo, para dar contexto
    completo cuando la carpeta inmediata no basta (ver caso Stanhopea/Clowesia)."""
    try:
        i = [p.upper() for p in partes_ruta].index("ESPECIE")
    except ValueError:
        return None
    return " / ".join(partes_ruta[i + 1:-1]) or None

def detectar_rama(partes_ruta):
    for p in partes_ruta:
        if p.upper() in ("AUTORES", "ESPECIE", "EVENTOS", "ESPACIOS"):
            return p.upper()
    return "(otra)"

print("Funciones de parseo listas (ya probadas contra ejemplos reales del árbol).")

In [ ]:
filas = []
for carpeta_raiz in CARPETAS_A_ESCANEAR:
    base = RAIZ_GALERIA / carpeta_raiz
    if not base.exists():
        print(f"Aviso: {base} no existe, se omite.")
        continue
    for archivo in base.rglob("*"):
        if archivo.is_dir():
            continue
        partes = archivo.relative_to(RAIZ_GALERIA).parts
        rama = detectar_rama(partes)
        carpeta_contenedora = archivo.parent.name
        autor_sug, conf_autor = (None, None)
        esp_cientifico, esp_comun, conf_especie = (None, None, None)

        breadcrumb_especie = None
        if rama == "AUTORES":
            ancla_autor = carpeta_autor_desde_partes(partes)
            autor_sug, conf_autor = parsear_autor(ancla_autor) if ancla_autor else (None, "baja")
        elif rama == "ESPECIE":
            esp_cientifico, esp_comun, conf_especie = parsear_especie(carpeta_contenedora)
            breadcrumb_especie = breadcrumb_desde_especie(partes)

        try:
            stat = archivo.stat()
            tam, f_creado, f_mod = stat.st_size, datetime.fromtimestamp(stat.st_ctime).isoformat(), datetime.fromtimestamp(stat.st_mtime).isoformat()
        except OSError as e:
            tam, f_creado, f_mod = None, None, None
            print(f"No se pudo leer: {archivo} ({e})")

        filas.append((
            str(archivo), archivo.name, archivo.suffix.lower(), carpeta_contenedora, rama,
            tam, f_creado, f_mod, autor_sug, esp_cientifico, esp_comun, conf_autor, conf_especie, breadcrumb_especie,
        ))

print(f"Archivos escaneados: {len(filas):,}")

In [ ]:
con.executemany("""
    INSERT OR REPLACE INTO staging_escaneo
    (ruta_completa, nombre_archivo, extension, carpeta_contenedora, rama_raiz,
     tamano_bytes, fecha_creacion, fecha_modificacion,
     autor_sugerido, especie_cientifico_sugerido, especie_comun_sugerido,
     confianza_autor, confianza_especie, breadcrumb_especie)
    VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)
""", filas)
con.commit()

total = con.execute("SELECT COUNT(*) FROM staging_escaneo").fetchone()[0]
print(f"Filas en staging_escaneo: {total:,}")

print("\nPor rama:")
for rama, n in con.execute("SELECT rama_raiz, COUNT(*) AS n FROM staging_escaneo GROUP BY rama_raiz ORDER BY n DESC"):
    print(f"  {rama:12s} {n:,}")

print("\nConfianza de autor (solo rama AUTORES):")
for conf, n in con.execute("SELECT confianza_autor, COUNT(*) FROM staging_escaneo WHERE rama_raiz='AUTORES' GROUP BY confianza_autor"):
    print(f"  {conf}: {n:,}")

print("\nConfianza de especie (solo rama ESPECIE):")
for conf, n in con.execute("SELECT confianza_especie, COUNT(*) FROM staging_escaneo WHERE rama_raiz='ESPECIE' GROUP BY confianza_especie"):
    print(f"  {conf}: {n:,}")

## Reporte para revisión manual

Exporta a Excel los casos de baja/media confianza, para que los revises antes de que sigamos con el triage hacia las tablas normalizadas.

In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

df_revisar = pd.read_sql('''
    SELECT rama_raiz, carpeta_contenedora, breadcrumb_especie, nombre_archivo, autor_sugerido, confianza_autor,
           especie_cientifico_sugerido, especie_comun_sugerido, confianza_especie, ruta_completa
    FROM staging_escaneo
    WHERE confianza_autor = 'baja' OR confianza_especie = 'media'
    ORDER BY rama_raiz, carpeta_contenedora
''', con)

wb = Workbook()
ws = wb.active
ws.title = "Para revisar"
header_font = Font(name="Arial", bold=True, color="FFFFFF")
header_fill = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
for c, h in enumerate(df_revisar.columns, start=1):
    cell = ws.cell(row=1, column=c, value=h); cell.font, cell.fill = header_font, header_fill
for r, row in enumerate(df_revisar.itertuples(index=False), start=2):
    for c, val in enumerate(row, start=1):
        ws.cell(row=r, column=c, value=str(val) if pd.notna(val) else "")
ws.freeze_panes, ws.auto_filter.ref = "A2", ws.dimensions
for i, w in enumerate([14, 35, 45, 40, 30, 12, 25, 25, 12, 90], start=1):
    ws.column_dimensions[get_column_letter(i)].width = w

ruta_reporte = Path("../salida") / "Volumen_Casos_Para_Revisar.xlsx"
ruta_reporte.parent.mkdir(exist_ok=True)
wb.save(ruta_reporte)
print(f"{len(df_revisar):,} filas para revisar. Guardado: {ruta_reporte.resolve()}")

con.close()